# Anima Scribe — QLoRA fine-tune on a free Colab T4

Fine-tunes **Qwen2.5 7B Instruct** (4-bit) into the Anima scribe voice — finished, literate replies — then DPO-sharpens it against the concrete failure modes (truncation, fragments, drift) and exports a GGUF for Ollama.

**Runtime → Change runtime type → T4 GPU** before running. Budget: ~60–90 min SFT, ~15 min DPO, ~15 min GGUF export.

You need two files produced on your machine by `pnpm llm:dataset` (they are gitignored):
`scripts/llm/output/finetune-sharegpt.jsonl`, `scripts/llm/output/finetune-sharegpt.val.jsonl`, and `scripts/llm/output/dpo-pairs.jsonl`. Upload them in step 3, or mount Drive and point at them.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print("bf16 supported:", torch.cuda.is_bf16_supported())  # False on T4 — the scripts pick fp16 automatically

## 1. Clone the repo and install the Unsloth stack

In [ ]:
%cd /content
!rm -rf Anima-Protocol && git clone --depth 1 https://github.com/davins56/Anima-Protocol.git
%cd /content/Anima-Protocol
!pip install -q "unsloth[colab-new]" transformers datasets trl
!python -m py_compile scripts/llm/finetune/unsloth_sft.py scripts/llm/finetune/unsloth_dpo.py scripts/llm/finetune/export_gguf.py && echo scripts ok

## 2. Bring the dataset

Either upload the three JSONL files here, or mount Drive and copy them into `scripts/llm/output/`.

In [ ]:
import os
os.makedirs("scripts/llm/output", exist_ok=True)
from google.colab import files
uploaded = files.upload()  # pick finetune-sharegpt.jsonl, finetune-sharegpt.val.jsonl, dpo-pairs.jsonl
for name in uploaded:
    os.replace(name, os.path.join("scripts/llm/output", name))
!wc -l scripts/llm/output/*.jsonl

## 3. SFT (QLoRA)

`--max-seq-len 2048` keeps a 7B model inside the T4's 15 GB with batch 2 × grad-accum 4. Raise `--epochs` to 2 if the train set is under ~600 rows and eval loss is still falling at the end of epoch 1.

In [ ]:
!python scripts/llm/finetune/unsloth_sft.py \
  --data scripts/llm/output/finetune-sharegpt.jsonl \
  --eval-data scripts/llm/output/finetune-sharegpt.val.jsonl \
  --base unsloth/Qwen2.5-7B-Instruct-bnb-4bit \
  --out scripts/llm/checkpoints/anima-scribe-qlora \
  --max-seq-len 2048 --epochs 1 --batch-size 2 --grad-accum 4

## 4. DPO on the scribe preference pairs

Sharpens the SFT adapter against truncation, fragment stacks, drift, list-dumps, and generic-assistant lapses.

In [ ]:
!python scripts/llm/finetune/unsloth_dpo.py \
  --data scripts/llm/output/dpo-pairs.jsonl \
  --base scripts/llm/checkpoints/anima-scribe-qlora \
  --out scripts/llm/checkpoints/anima-scribe-dpo \
  --max-seq-len 2048 --epochs 1

## 5. Quick sanity chat before exporting

Every reply should end on a finished sentence. If they still trail off, the base model was not the problem — check the dataset stats.

In [ ]:
from unsloth import FastLanguageModel
import re
model, tok = FastLanguageModel.from_pretrained("scripts/llm/checkpoints/anima-scribe-dpo", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
system = open("scripts/llm/Modelfile.anima-scribe").read().split('SYSTEM """')[1].split('"""')[0]
prompts = [
    "Explain, honestly and in full, what it means when you say you remember me.",
    "I got the exact parking spot I wanted. Best day of my life.",
    "You once said devotion survives examination. Prove it, without hiding behind an image.",
]
for p in prompts:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": p}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=400, temperature=0.7, top_p=0.9, repetition_penalty=1.12, do_sample=True)
    reply = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    finished = bool(re.search(r'[.!?…]["\'”’)]*$', reply))
    print(f"\n### {p}\n{reply}\n--> finished sentence: {finished}, {len(reply.split())} words")

## 6. Export GGUF for Ollama

Writes `scripts/llm/gguf/anima-scribe-q4_k_m.gguf` (~4.7 GB). Then download it or copy to Drive.

In [ ]:
!python scripts/llm/finetune/export_gguf.py \
  --adapter scripts/llm/checkpoints/anima-scribe-dpo \
  --out scripts/llm/gguf --prefix anima-scribe --quant q4_k_m
!ls -lh scripts/llm/gguf

In [ ]:
# Option A — copy to Google Drive (recommended for a 4.7 GB file)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/anima-llm && cp scripts/llm/gguf/anima-scribe-q4_k_m.gguf /content/drive/MyDrive/anima-llm/
# Option B — direct download
# from google.colab import files; files.download("scripts/llm/gguf/anima-scribe-q4_k_m.gguf")

## 7. On the Ollama host

```bash
mkdir -p scripts/llm/gguf && mv ~/Downloads/anima-scribe-q4_k_m.gguf scripts/llm/gguf/
ollama create anima-scribe -f scripts/llm/Modelfile.anima-scribe-tuned
export ANIMA_OLLAMA_MODEL_STANDARD=anima-scribe
pnpm llm:eval        # complete-thought-* cases must pass
```

Then set `ANIMA_OLLAMA_MODEL_STANDARD=anima-scribe` on the production host (Fly / VPS) and redeploy — see `docs/custom-llm.md`.